In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model='gpt-4o-mini')

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool
def get_current_time(timezone:str, location: str) -> str:
    """
    현재 시각을 반환하는 함수

    Args:
        timezone(str) : 타임존(예: 'Asia/Seoul'). 실제 존재해야함
        location(str) : 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨.
    """

    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재 시각 {now}'
    print(location_and_local_time)
    return location_and_local_time

In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time]
tool_dict = {"get_current_time": get_current_time}

# 도구를 모델에 바인딩
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?")
]

response = llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6b1e1d7ece', 'id': 'chatcmpl-DeZLneKS8PyLd6RQguQwyGFcQmmmZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1a84-e501-7673-a710-efc1f72c8d6f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': 'Busan'}, 'id': 'call_Zl03YIC1K8kMkSquFEcdkdE5', 'type': 'tool_call'

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': 'Busan'}
Asia/Seoul (Busan) 현재 시각 2026-05-12 13:51:56


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6b1e1d7ece', 'id': 'chatcmpl-DeZLneKS8PyLd6RQguQwyGFcQmmmZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1a84-e501-7673-a710-efc1f72c8d6f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': 'Busan'}, 'id': 'call_Zl03YIC1K8kMkSquFEcdkdE5', 'type': 'tool_cal

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 5월 12일 13시 51분 56초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 186, 'total_tokens': 212, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6b1e1d7ece', 'id': 'chatcmpl-DeZOGGxMcVTCp4ncHzyMMomeHSSox', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1a87-3fd4-7973-a8a6-2c5f398d54cd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 186, 'output_tokens': 26, 'total_tokens': 212, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field


class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title='주식 코드', description='주식 코드 (예: AAPL)')
    period: str = Field(..., title='기간', description='주식 데이터 조회 기간 (예: 1d, 1mo, 1y)')
    

In [8]:
import yfinance as yf


@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time" : get_current_time, "get_yf_stock_history":get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 277, 'total_tokens': 304, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7b0b5e194', 'id': 'chatcmpl-DeZf9CODdAiS0CkigkoEqmrCiPuZX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e1a97-3768-7550-a7c5-49c532e143dc-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_ulNlzlZJEZlrO4Bi0EvXvuwb', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 277, 'output_tokens': 27, 'total_tokens': 304, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [11]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-04-13 00:00:00-04:00 | 350.07 | 356.35 | 348.57 |  352.42 | 5.36175e+07 |           0 |              0 |\n| 2026-04-14 00:00:00-04:00 | 357.67 | 367.63 | 354.77 |  364.2  | 5.99796e+07 |           0 |              0 |\n| 2026-04-15 00:00:00-04:00 | 366.83 | 394.65 | 362.5  |  391.95 | 1.1381e+08  |           0 |              0 |\n| 2026-04-16 00:00:00-04:00 | 393.81 | 394.06 | 381.8  |  388.9  | 6.35151e+07 |           0 |              0 |\n| 2026-04-17 00:00:00-04:00 | 395.92 | 409.28 | 391.65 |  400.62 | 9.064e+07   |           0 |              0 |\n| 2026-04-20 00:00:00-04:00 | 402.58 | 406.8  | 388.33 |  392.5  | 6.46039e+07 |           0 |              0 |\n| 2026-04-21 00:00:00-04:0

In [12]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전인 2026년 4월 11일 테슬라(TSLA)의 주가는 약 $352.42였고, 현재(2026년 5월 11일)의 주가는 $445입니다. \n\n따라서, 테슬라의 주가는 한 달 전에 비해 올랐습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 1581, 'total_tokens': 1652, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7b0b5e194', 'id': 'chatcmpl-DeZhMFqGOJvPkbJoZwRWBQ2PLAC6t', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1a99-50b2-7633-baab-c07887a667e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1581, 'output_tokens': 71, 'total_tokens': 1652, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

is_first = True
for chunk in response:
    print('chunk type: ', type(chunk))

    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk

    print('content: ', gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'type': 'tool_c

In [14]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f9748774de', 'service_tier': 'default'}, id='lc_run--019e1aad-6210-7da3-be84-7bc465347676', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 23, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [15]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2026-05-12 14:35:10


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f9748774de', 'service_tier': 'default'}, id='lc_run--019e1aad-6210-7da3-be84-7bc465347676', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 23, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_X4bLQwRojn15xa6b3NwhKi97', 'index': 0, 'ty

In [16]:
for c in llm_with_tools.stream(messages):
    print(c.content, end="|")

|부|산|은| 지금| |202|6|년| |5|월| |12|일| |14|시| |35|분|입니다|.||||